In [ ]:
# 1. Setup Kaggle
!pip install -q kaggle
from google.colab import files
import os

# Upload your kaggle.json file
print("Please upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 2. Download and Unzip
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip
print("Data ready!")

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
100% 60.3M/60.3M [00:00<00:00, 184MB/s]

Data ready!


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    'train',
    target_size=(48, 48),
    batch_size=64,
    color_mode="grayscale",
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    'train',
    target_size=(48, 48),
    batch_size=64,
    color_mode="grayscale",
    class_mode='categorical',
    subset='validation'
)

Found 22968 images belonging to 7 classes.
Found 5741 images belonging to 7 classes.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization

def build_model():
    model = Sequential()
    model.add(Conv2D(32, (3,3), activation='relu', input_shape=(48,48,1)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))

    model.add(Conv2D(64, (3,3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))

    model.add(Flatten())
    model.add(Dense(512, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(7, activation='softmax'))
    return model

model = build_model()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 46, 46, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 21, 21, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     3,277,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,302,151 (12.60 MB)

 Trainable params: 3,300,935 (12.59 MB)

 Non-trainable params: 1,216 (4.75 KB)

In [ ]:
# 1. Define how many "rounds" (epochs) the model should train
EPOCHS = 25

# 2. Start the training process
print("Starting training on GPU... Fasten your seatbelts!")
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // 64,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=val_generator.samples // 64
)

# 3. Save the trained weights to a file
model.save('emotion_model.h5')
print("\nTraining Complete! The model has been saved as 'emotion_model.h5'.")

Starting training on GPU... Fasten your seatbelts!
Epoch 1/25
358/358 ━━━━━━━━━━━━━━━━━━━━ 147s 401ms/step - accuracy: 0.2746 - loss: 2.1389 - val_accuracy: 0.2384 - val_loss: 3.0034
Epoch 2/25
  1/358 ━━━━━━━━━━━━━━━━━━━━ 3:13 541ms/step - accuracy: 0.1719 - loss: 1.8670

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


358/358 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.1719 - loss: 1.8670 - val_accuracy: 0.2395 - val_loss: 3.0214
Epoch 3/25
358/358 ━━━━━━━━━━━━━━━━━━━━ 191s 400ms/step - accuracy: 0.3598 - loss: 1.6775 - val_accuracy: 0.3711 - val_loss: 1.5986
Epoch 4/25
358/358 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.4062 - loss: 1.6163 - val_accuracy: 0.3799 - val_loss: 1.5790
Epoch 5/25
358/358 ━━━━━━━━━━━━━━━━━━━━ 141s 394ms/step - accuracy: 0.3976 - loss: 1.5545 - val_accuracy: 0.3766 - val_loss: 1.5944
Epoch 6/25
358/358 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.3906 - loss: 1.6962 - val_accuracy: 0.3892 - val_loss: 1.5852
Epoch 7/25
358/358 ━━━━━━━━━━━━━━━━━━━━ 262s 584ms/step - accuracy: 0.4234 - loss: 1.4923 - val_accuracy: 0.3160 - val_loss: 2.1404
Epoch 8/25
358/358 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.2969 - loss: 1.5864 - val_accuracy: 0.3229 - val_loss: 2.0025
Epoch 9/25
358/358 ━━━━━━━━━━━━━━━━━━━━ 138s 386ms/step - accuracy: 0.4351 - loss: 1.4666 - val_a


Training Complete! The model has been saved as 'emotion_model.h5'.


In [ ]:
from google.colab import files
files.download('emotion_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>